## First read and open the Reseller.csv

In [0]:
df = spark.read.csv("/Volumes/workspace/default/project/Reseller.csv", sep=[",/t"] , header=True , inferSchema=True)

display(df)

## All the values are in the fisrt column . Only one column exists

In [0]:
df.describe()

In [0]:
df.printSchema()

## Distribute values in the multiple columns

In [0]:
from pyspark.sql.functions import split, col, trim

#
raw_df = spark.read.text("/Volumes/workspace/default/project/Reseller.csv")

# 2. Χρησιμοποιούμε regex στο split: το [,\t]+ σημαίνει "ένα ή περισσότερα κόμματα ή tabs"
# Αυτό θα δημιουργήσει μια στήλη τύπου array
df_split = raw_df.withColumn("cols", split(col("value"), r"[,\t]+"))

# 3. Επιλέγουμε τα στοιχεία του array και τα αντιστοιχίζουμε στις στήλες σου
# Χρησιμοποιούμε trim() για να καθαρίσουμε τυχόν εναπομείναντα κενά
final_df = df_split.select(
    trim(col("cols").getItem(0)).alias("ResellerKey"),
    trim(col("cols").getItem(1)).alias("Business Type"),
    trim(col("cols").getItem(2)).alias("Reseller"),
    trim(col("cols").getItem(3)).alias("City"),
    trim(col("cols").getItem(4)).alias("State-Province"),
    trim(col("cols").getItem(5)).alias("Country-Region")
)

# 4. Εμφάνιση αποτελέσματος
display(final_df)

## Remove the first row from the final_df

In [0]:
# Φιλτράρει τη σειρά όπου το ResellerKey έχει την τιμή της κεφαλίδας
final_df = final_df.filter(col("Resellerkey") != "ResellerKey")

display(final_df)

## Change data type in ResellerKey

In [0]:
from pyspark.sql.functions import col

# Μετατροπή μιας στήλης
final_df = final_df.withColumn("ResellerKey", col("ResellerKey").cast("int"))

# Εμφάνιση του νέου σχήματος για επιβεβαίωση
final_df.printSchema()



In [0]:
display(final_df)

## Create a Temporary view for data inspection

In [0]:
final_df.createOrReplaceTempView("Reseller_table")


In [0]:
#try to find different formats per column
display(spark.sql("select distinct `Business Type` from Reseller_table"))

In [0]:
display(spark.sql("select distinct ResellerKey from Reseller_table order by ResellerKey"))

In [0]:
display(spark.sql("select distinct Reseller from Reseller_table order by Reseller"))

In [0]:
display(spark.sql('select ResellerKey, row_number() over (partition by ResellerKey order by ResellerKey) as rn from Reseller_table order by ResellerKey'))

In [0]:
#test for git